<a href="https://colab.research.google.com/github/Rafa11-ai/data-science-2024/blob/main/Pertemuan12_Rafa_Hidayah_240401010203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama  : Rafa Hidayah

NIM   : 240401010203

Kelas : IF405

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
  n_item = np.random.randint(2, 6)
  transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
  if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
    transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


In [ ]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [ ]:
from mlxtend.frequent_patterns import apriori
for ms in [0.05, 0.1, 0.2]:
  freq = apriori(df, min_support=ms, use_colnames=True)
  print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence',
                          min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents',
             'support', 'confidence', 'lift']].head(10))

# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

         antecedents consequents  support  confidence      lift
10       (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
15  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
12      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
13     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
9      (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
14   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


Aturan yang paling kuat adalah aturan dengan nilai Lift tertinggi. Karena data sintetis pada modul sengaja dibuat dengan pola Roti sering bersama Selai, aturan yang melibatkan Roti → Selai diharapkan menjadi salah satu aturan yang kuat.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
                 'Dairy','Minuman','Bumbu','Minuman','Dairy']
    })
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
  idx = katalog.index[katalog['produk'] == nama_produk][0]
  skor = list(enumerate(sim_matrix[idx]))
  skor = sorted(skor, key=lambda x: x[1], reverse=True)
  skor = [s for s in skor if s[0] != idx][:top_n]
  return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()
print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


In [ ]:
produk_target = 'Roti'
# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(
    lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


Apakah kedua pendekatan memberi rekomendasi yang konsisten?
-

Kedua pendekatan belum tentu memberikan rekomendasi yang sama karena menggunakan dasar perhitungan yang berbeda. Association Rule Mining menghasilkan rekomendasi berdasarkan pola pembelian aktual dalam transaksi, sedangkan Content-Based Filtering memberikan rekomendasi berdasarkan kemiripan kategori produk. Oleh karena itu, apabila keduanya memberikan produk yang sama, rekomendasi tersebut dapat dianggap lebih kuat karena didukung oleh pola pembelian dan kesamaan karakteristik produk.



Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?
-

Association Rules lebih tepat digunakan ketika tersedia data transaksi yang cukup dan tujuan utamanya adalah menemukan pola produk yang sering dibeli bersama. Content-Based Filtering lebih sesuai ketika informasi atau atribut produk tersedia dengan baik dan ingin memberikan rekomendasi berdasarkan kemiripan produk. Kedua pendekatan dapat digabungkan menjadi hybrid recommendation system untuk memanfaatkan kelebihan masing-masing dan mengurangi keterbatasan satu metode. Modul juga menjelaskan bahwa sistem rekomendasi produksi umumnya menggabungkan beberapa pendekatan.

Yang dipelajari: menemukan pola pembelian menggunakan Apriori/Association Rules dan membuat rekomendasi menggunakan Content-Based Filtering.

Temuan utama: aturan terkuat adalah (Teh, Keju) → (Telur) dengan confidence 0,857 dan lift 2,381. Untuk produk Roti, kedua pendekatan sama-sama merekomendasikan Selai, meskipun Content-Based juga memberikan Sereal dan Susu.

Keterbatasan/pertanyaan: dataset hanya terdiri dari 50 transaksi sintetis, sehingga pola yang ditemukan belum tentu berlaku pada transaksi nyata. Pertanyaannya adalah apakah pola tersebut tetap kuat jika jumlah dan variasi transaksi diperbesar.